# Notebook 26 — Great Britain race-population completeness

## Audit question

**Is the Great Britain race population in immutable Source Version 1 materially compromised in 2020, with 2019 and 2021 serving as controls?**

This notebook is a database/source-correctness investigation discovered while Great Britain Study 05 was examining sex restrictions. Study 05 remains paused at its pushed checkpoint while this audit is resolved.

### Working hypothesis

> **H₁: the Source Version 1 Great Britain race-population defect is concentrated in 2020.**

This is a hypothesis to test, not a conclusion.

The investigation therefore starts with **2019 / 2020 / 2021**, using 2019 and 2021 as controls. It does **not** begin with a uniform 2015–2026 audit.

If the controls are clean and 2020 is anomalous, the next stage will localise the 2020 discrepancy by month and racecourse. Additional surrounding years will then be used as falsification controls.


## Evidence boundary and standing rules

- **British Horseracing Authority (BHA) data is the primary source for the official British race population.**
- Immutable Source Version 1 and accepted Database v4 remain read-only.
- The prohibited **Source Version 1 off-time field must not be queried, selected, loaded, displayed, sorted on, derived from or used for reconciliation anywhere in this investigation.**
- When individual-race reconciliation becomes necessary, use governed/project-owned Inside Rails identities and governed racecourse identity.
- Fixture/date/course race counts are screening evidence only. A count mismatch is a **candidate population discrepancy** until investigated.
- A BHA fixture record is not itself evidence that races were run: the BHA fixture endpoint can return records whose race-list endpoint contains zero races. Official race counts must therefore come from the fixture race-list records.
- Every substantive result must be followed immediately by a written finding before moving to the next analytical question.
- This notebook must fail closed rather than silently switching source/database artefacts.

### Known positive control entering this audit

The Study 05 checkpoint already established the following defect for **6 June 2020**:

| Source | Newcastle | Newmarket | Lingfield Park | GB races |
|---|---:|---:|---:|---:|
| Official BHA race lists | 10 | 9 | 9 | 28 |
| Immutable Source Version 1 | 10 | 0 | 0 | 10 |
| Database v4 | 10 | 0 | 0 | 10 |

Source Version 1 also contains two South African races on that date.

The upstream tracing is already complete: the 18 Newmarket/Lingfield races are absent from Source Version 1 itself, and Database v4 inherited that omission.

Notebook 26 uses this date as a **positive-control sanity check** before testing the wider 2019/2020/2021 hypothesis.


In [ ]:
from pathlib import Path
import hashlib
import json
import re
import time
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import pandas as pd

from inside_rails.source_sqlite import connect_read_only


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    while current != current.parent:
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    raise RuntimeError("Could not locate Inside Rails project root")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


PROJECT_ROOT = find_project_root(Path.cwd())

SOURCE_VERSION_1 = (
    PROJECT_ROOT
    / "data/raw/form_2015-present/form_2015-present/raceform.db"
)

DATABASE_V4 = (
    PROJECT_ROOT
    / "data/processed/database/releases/inside_rails_v4.sqlite3"
)

EXPECTED_SOURCE_V1_SHA256 = (
    "77b5dbbbfdee69d4d92a582655344e1e5ba29ca4646a5999c383de8161eeeaa7"
)

EXPECTED_DATABASE_V4_SHA256 = (
    "45ad0c3d81d457385d655d9c47b030c5815c638e477281a9be8aabf164eecff7"
)

assert SOURCE_VERSION_1.is_file(), (
    f"Accepted Source Version 1 not found: {SOURCE_VERSION_1}"
)

assert DATABASE_V4.is_file(), (
    f"Accepted Database v4 not found: {DATABASE_V4}"
)

source_v1_sha256 = sha256_file(SOURCE_VERSION_1)
database_v4_sha256 = sha256_file(DATABASE_V4)

assert source_v1_sha256 == EXPECTED_SOURCE_V1_SHA256, (
    "Source Version 1 SHA-256 mismatch: "
    f"expected {EXPECTED_SOURCE_V1_SHA256}, observed {source_v1_sha256}"
)

assert database_v4_sha256 == EXPECTED_DATABASE_V4_SHA256, (
    "Database v4 SHA-256 mismatch: "
    f"expected {EXPECTED_DATABASE_V4_SHA256}, observed {database_v4_sha256}"
)

print("Project root:", PROJECT_ROOT)
print("Source Version 1:", SOURCE_VERSION_1)
print("Source Version 1 SHA-256:", source_v1_sha256)
print("Database v4:", DATABASE_V4)
print("Database v4 SHA-256:", database_v4_sha256)


### Execution checkpoint — exact artefact identity

Do not continue unless the preceding cell completes without assertion failure and prints the documented SHA-256 identities for both immutable artefacts.

No Source Version 1 table or field is queried by that gate; it verifies only the file identity.

For the initial population screen, Database v4's governed GB race-occurrence view will be used as the source-backed comparator because it preserves one row per admitted GB source race occurrence while supplying project-owned race occurrence and racecourse identities. Direct Source Version 1 inspection is unnecessary for the first screening stage and would add no independent population information because the known defect has already been traced upstream.

## Next question

Can the audit reproduce the known **6 June 2020** BHA-versus-source-backed population discrepancy without using the prohibited source off-time field?


In [ ]:
# Acquire the current public BHA frontend Authorization value.
#
# This reuses the acquisition method established in Study 05.
# The value is held in memory only and is never printed or persisted.

BHA_API_ROOT = "https://api09.horseracing.software"

BHA_APP_JS = (
    "https://www.britishhorseracing.com/"
    "wp-content/themes/bha/library/js/angular/app.js"
)

USER_AGENT = (
    "Mozilla/5.0 (X11; Linux x86_64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/131.0 Safari/537.36"
)

app_request = Request(
    BHA_APP_JS,
    headers={"User-Agent": USER_AGENT},
)

with urlopen(app_request, timeout=30) as response:
    app_js_text = response.read().decode("utf-8", errors="replace")

authorization_pattern = re.compile(
    r"""
    \$httpProvider
    \.defaults
    \.headers
    \.common
    \[['"]Authorization['"]\]
    \s*=\s*
    ['"]([^'"]+)['"]
    """,
    re.VERBOSE,
)

authorization_matches = []

for line in app_js_text.splitlines():
    stripped = line.lstrip()

    if stripped.startswith("//"):
        continue

    match = authorization_pattern.search(line)

    if match:
        authorization_matches.append(match.group(1))

assert len(authorization_matches) == 1, (
    "Expected exactly one active BHA Authorization assignment; "
    f"found {len(authorization_matches)}."
)

authorization_value = authorization_matches[0]


def get_bha_json(url: str) -> dict:
    request = Request(
        url,
        headers={
            "Authorization": authorization_value,
            "Accept": "application/json",
            "Origin": "https://www.britishhorseracing.com",
            "Referer": "https://www.britishhorseracing.com/",
            "User-Agent": USER_AGENT,
        },
    )

    with urlopen(request, timeout=30) as response:
        return json.loads(response.read().decode("utf-8"))


print("BHA acquisition helper ready.")


In [ ]:
# Positive control: official BHA race population for 6 June 2020.
#
# Important:
# - fixture rows are not counted as races;
# - every fixture's race-list endpoint is inspected;
# - zero-race fixture records remain visible in the screening output;
# - no race-time field is required for this population count.

POSITIVE_CONTROL_DATE = "2020-06-06"

fixture_params = {
    "fromdate": POSITIVE_CONTROL_DATE,
    "todate": POSITIVE_CONTROL_DATE,
    "resultsAvailable": 1,
    "order": "asc",
    "page": 1,
    "per_page": 100,
}

fixture_url = (
    f"{BHA_API_ROOT}/bha/v1/fixtures/?"
    + urlencode(fixture_params)
)

fixture_payload = get_bha_json(fixture_url)
bha_fixtures = fixture_payload["data"]

bha_fixture_race_counts = []
bha_race_rows = []

for fixture in bha_fixtures:
    race_list_url = (
        f"{BHA_API_ROOT}/bha/v1/fixtures/"
        f"{fixture['fixtureYear']}/"
        f"{fixture['fixtureId']}/races"
    )

    race_payload = get_bha_json(race_list_url)
    races = race_payload["data"]

    bha_fixture_race_counts.append(
        {
            "fixture_year": fixture.get("fixtureYear"),
            "fixture_id": fixture.get("fixtureId"),
            "course_name": fixture.get("courseName"),
            "races": len(races),
        }
    )

    for race in races:
        bha_race_rows.append(
            {
                "race_date": race.get("raceDate"),
                "course_name": fixture.get("courseName"),
                "fixture_id": fixture.get("fixtureId"),
                "bha_race_id": race.get("raceId"),
                "division_sequence": race.get("divisionSequence"),
                "race_name": race.get("raceName"),
            }
        )

    # Retain the gentle pacing used during Study 05 to avoid unnecessary
    # pressure on the public BHA service.
    time.sleep(1.5)

bha_fixture_race_counts = pd.DataFrame(bha_fixture_race_counts)
bha_races = pd.DataFrame(bha_race_rows)

bha_active_fixture_counts = (
    bha_fixture_race_counts
    .loc[bha_fixture_race_counts["races"] > 0, ["course_name", "races"]]
    .sort_values("course_name")
    .reset_index(drop=True)
)

expected_bha_counts = {
    "Lingfield Park": 9,
    "Newcastle": 10,
    "Newmarket": 9,
}

observed_bha_counts = dict(
    zip(
        bha_active_fixture_counts["course_name"],
        bha_active_fixture_counts["races"],
    )
)

assert observed_bha_counts == expected_bha_counts, (
    "Positive-control BHA course counts changed: "
    f"expected {expected_bha_counts}, observed {observed_bha_counts}"
)

assert len(bha_races) == 28, (
    f"Expected 28 official BHA races, observed {len(bha_races)}"
)

print("BHA fixture records returned:", len(bha_fixture_race_counts))
print("BHA race records returned:", len(bha_races))
print()
print("BHA race counts by fixture record:")
display(
    bha_fixture_race_counts[
        ["course_name", "fixture_id", "races"]
    ]
)
print()
print("BHA active race counts:")
display(bha_active_fixture_counts)


In [ ]:
# Positive control: source-backed Database v4 population on 6 June 2020.
#
# This deliberately selects only project-owned/governed fields needed for the
# population check. The prohibited Source Version 1 off-time field is neither
# queried nor exposed.
#
# We inspect:
#   1. the governed GB race-occurrence view;
#   2. the general reconciled race-occurrence view to confirm the two non-GB
#      source races already known from Study 05.

with connect_read_only(DATABASE_V4) as connection:

    v4_gb_positive_control = pd.read_sql_query(
        """
        SELECT
            source_race_occurrence_code,
            raw_date AS race_date,
            governed_racecourse_name
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date = ?
        ORDER BY
            governed_racecourse_name,
            source_race_occurrence_code
        """,
        connection,
        params=[POSITIVE_CONTROL_DATE],
    )

    v4_all_positive_control = pd.read_sql_query(
        """
        SELECT
            source_race_occurrence_code,
            raw_date AS race_date,
            candidate_jurisdiction,
            candidate_course_label
        FROM view_reconciled_race_occurrences
        WHERE raw_date = ?
        ORDER BY
            candidate_jurisdiction,
            candidate_course_label,
            source_race_occurrence_code
        """,
        connection,
        params=[POSITIVE_CONTROL_DATE],
    )

v4_gb_course_counts = (
    v4_gb_positive_control
    .groupby("governed_racecourse_name", as_index=False)
    .size()
    .rename(columns={"size": "races"})
)

v4_jurisdiction_counts = (
    v4_all_positive_control
    .groupby("candidate_jurisdiction", as_index=False)
    .size()
    .rename(columns={"size": "races"})
)

assert len(v4_gb_positive_control) == 10, (
    f"Expected 10 source-backed GB races, observed {len(v4_gb_positive_control)}"
)

assert v4_gb_course_counts.to_dict("records") == [
    {"governed_racecourse_name": "Newcastle", "races": 10}
], (
    "Expected the source-backed GB positive control to contain only "
    f"10 Newcastle races; observed {v4_gb_course_counts.to_dict('records')}"
)

assert len(v4_all_positive_control) == 12, (
    "Expected 12 source-backed races of all jurisdictions on the date; "
    f"observed {len(v4_all_positive_control)}"
)

print("Source-backed Database v4 GB races:", len(v4_gb_positive_control))
display(v4_gb_course_counts)
print()
print("Source-backed Database v4 races by jurisdiction:")
display(v4_jurisdiction_counts)


### Finding — 6 June 2020 is the audit's positive control

The checkpoint evidence entering Notebook 26 establishes the expected contrast:

- official BHA race lists: **28 GB races**;
- source-backed Database v4 GB population: **10 races**;
- missing from the source-backed population: **18 races**;
- missing fixtures: **Newmarket (9)** and **Lingfield Park (9)**;
- Newcastle is present with the correct **10-race** count;
- the general source-backed date population also contains the previously identified **2 South African races**.

The two preceding cells are a fail-loud reproducibility check of that known state. They deliberately avoid the prohibited Source Version 1 off-time field.

This positive control establishes that the screening machinery can detect a known population omission while preserving the audit's source-field prohibition.

## Next bounded question

> **Across 2019, 2020 and 2021, do official BHA race-list counts and the source-backed Inside Rails GB race population agree in the two control years while diverging materially in 2020?**

The next stage should remain a **screening comparison**. It should first establish annual/date/course count patterns. Any mismatch found there remains a candidate discrepancy until reconciled; it must not immediately be labelled a genuine missing race.
